# CROSCIM UNet (Supervised, SIT/UOAI forecast) — Interactive Test Notebook

Notebook pour charger le datamodule `*_simplify` (dataloader "test_simple"),
exécuter le forward pass du modèle UNet supervisé et explorer les prédictions
`pred_SIT` vs la cible `models_SIT`, pour diagnostiquer visuellement les
prédictions "étranges" observées avec `xp=CROSCIM/UNet_solvers/base_arctic_croscim_test_sit_UOAI_supervised_forecast`.

Adapté de `Notebook_CROSCIM_test_simplify.ipynb` (modèle de consistance) — ici
pour le solveur UNet déterministe (`UNetModel3`), donc sans swap EMA ni
échantillonnage d'ensemble stochastique (non applicables à un modèle
déterministe).

**Flux** :
1. Setup environnement & imports
2. Chargement config (xp `wpreproc` — même modèle/solveur, dataloader `_simplify`) + datamodule
3. Chargement checkpoint modèle
4. Itération sur les batchs de test → collecte predictions/targets
5. Cartes spatiales prédiction vs vérité (`pred_SIT` vs `models_SIT`)
6. Cartes d'erreur / résidu
7. Séries temporelles & métriques scalaires
8. Diagnostic — dataloader brut (xp originale) en x50 : `cristal_SIT` + inférence `pred_SIT`

## 1. Imports & Environment Setup

In [ ]:
import sys, os

# ── Project root on path ──────────────────────────────────────────────
ROOT = "/Odyssey/private/m19beauc/4dvarnet-starter"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore")

import copy
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
from IPython.display import display

import pytorch_lightning as pl
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

# ── Register custom OmegaConf resolvers (auto-registered on import)
import src.resolvers  # noqa: registers 'python_eval' resolver via module-level call

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"PyTorch Lightning: {pl.__version__}")


## 2. Load Configuration & Instantiate DataModule

In [ ]:
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate
from omegaconf import OmegaConf

# ── Config path ───────────────────────────────────────────────────────
# Same model/solver as the raw `test_sit_UOAI_supervised_forecast` xp, but
# already wired to the `_simplify` (test_simple) dataloader — reads the
# pre-built preproc_CROSCIM_x{50,10}.nc files instead of raw CIMR/CRISTAL
# files, bypassing domain_limits/mask/reference_source entirely. Useful to
# check whether "étranges" predictions come from the raw-dataloader patch
# construction or from the model/training itself.
CONFIG_DIR = os.path.join(ROOT, "config")
XP_NAME    = "CROSCIM/UNet_solvers/base_arctic_croscim_wpreproc_sit_UOAI_supervised_forecast"
CKPT_PATH  = "/Odyssey/private/m19beauc/4dvarnet-starter/ckpt/CROSCIM/base_croscim_UNet_sit_UOAI_supervised_forecast.ckpt"
DOMAIN     = "arctic_croscim"

# ── Load with Hydra compose API ───────────────────────────────────────
# 'domain' is a config GROUP (in the defaults list of main.yaml), not a plain key.
# Select it the same way as xp: just 'domain=<name>' (no + prefix).
# Clear GlobalHydra in case a previous cell already initialised it.
from hydra.core.global_hydra import GlobalHydra
GlobalHydra.instance().clear()

with initialize_config_dir(config_dir=CONFIG_DIR, version_base=None):
    cfg = compose(
        config_name="main",
        overrides=[
            f"xp={XP_NAME}",
            f"domain={DOMAIN}",   # config group override, no +
        ],
    )

print(OmegaConf.to_yaml(cfg.datamodule, resolve=True))


In [ ]:
# ── Instantiate & setup datamodule ───────────────────────────────────
dm = instantiate(cfg.datamodule)
dm.setup("test")

test_dl = dm.test_dataloader()
print(f"Test batches  : {len(test_dl)}")
print(f"Batch size    : {test_dl.batch_size}")
print(f"Resolutions   : {dm.multires}")
print(f"Target vars   : {dm.target_vars}")
print(f"Input vars    : {dm.input_vars}")

# Peek at one batch to inspect shapes
sample_batch = next(iter(test_dl))
print("\nBatch keys:", list(sample_batch.keys()) if isinstance(sample_batch, dict) else type(sample_batch))
if isinstance(sample_batch, dict):
    for key, item in sample_batch.items():
        print(f"  {key}:")
        for field in item._fields:
            val = getattr(item, field)
            if isinstance(val, torch.Tensor):
                print(f"    {field}: {tuple(val.shape)}")


## 3. Load Trained Model Checkpoint

In [ ]:
from contrib.CROSCIM.models.models_supervised import Lit4dVarNet_CROSCIM_Supervised

# ── Instantiate model directly via Hydra (handles python_eval interpolations) ──
# OmegaConf.to_container(resolve=True) fails on custom resolvers like python_eval.
# hydra.utils.instantiate resolves everything correctly inside the Hydra context.
model = instantiate(cfg.model)

# ── Load checkpoint weights ───────────────────────────────────────────
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    state = ckpt.get("state_dict", ckpt)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"✅ Checkpoint loaded: {CKPT_PATH}")
    print(f"   Missing keys    : {len(missing)}")
    print(f"   Unexpected keys : {len(unexpected)}")
else:
    print(f"⚠️  Checkpoint not found at {CKPT_PATH}. Running with random weights.")

model = model.to(DEVICE)
model.eval()
print(f"\nModel on {DEVICE}, eval mode.")


## 4. Run Forward Pass Over Test Batches & Collect Outputs

In [ ]:
N_BATCHES = None   # set to an int (e.g. 10) to limit the number of batches processed

# Storage: list of dicts per resolution
results = {res: [] for res in model.multires}

def _find_obs_field(batch, var_suffix):
    """
    Find the satellite observation field for a given variable suffix.
    Looks for batch fields that contain var_suffix but are NOT prefixed by 'models_'.
    E.g. for 'SIT' → returns 'cristal_SIT' or 'cimr_SIT' (first match).
    Falls back to var_mapping lookup if no direct match is found.
    """
    for field in batch._fields:
        if var_suffix in field and not field.startswith("models_"):
            val = getattr(batch, field)
            if isinstance(val, torch.Tensor):
                return field
    return None

model.eval()
with torch.no_grad():
    for batch_idx, batch in enumerate(test_dl):
        if N_BATCHES is not None and batch_idx >= N_BATCHES:
            break

        for res in model.multires:
            res_key = f"patch_x{res}"

            # Extract the resolution-specific NamedTuple from the dict batch
            b = batch[res_key] if isinstance(batch, dict) else batch
            b = model.modify_batch(b, res)

            # Forward pass
            sbatch = model.format_batch_for_solver(b, include_masks=model.include_masks, res=res)
            out_tensor = model(batch=sbatch, res=res)
            out = model.split_tensor_to_dict(out_tensor, res=res)

            # Collect predictions & targets on CPU
            entry = {"batch_idx": batch_idx, "res": res}
            tgt_vars = model._get_target_vars_for_resolution(res)
            for var in tgt_vars:
                var_suffix = var.split("_", 1)[1] if "_" in var else var
                pred = out[f"pred_{var_suffix}"].cpu().float()
                tgt  = getattr(b, var).cpu().float()

                # Find obs field: scan batch fields for var_suffix not starting with 'models_'
                obs_field = _find_obs_field(b, var_suffix)
                obs = getattr(b, obs_field).cpu().float() if obs_field else None

                entry[f"pred_{var_suffix}"] = pred
                entry[f"tgt_{var_suffix}"]  = tgt
                if obs is not None:
                    entry[f"obs_{var_suffix}"] = obs

            results[res].append(entry)

        if batch_idx % 5 == 0:
            print(f"  batch {batch_idx:3d} done")

# Print obs fields found for first batch
for res in model.multires:
    tgt_vars = model._get_target_vars_for_resolution(res)
    res_key = f"patch_x{res}"
    b0 = next(iter(test_dl))
    b0 = b0[res_key] if isinstance(b0, dict) else b0
    b0 = model.modify_batch(b0, res)
    for var in tgt_vars:
        var_suffix = var.split("_", 1)[1] if "_" in var else var
        obs_field = _find_obs_field(b0, var_suffix)
        print(f"  res=x{res}  {var_suffix}  →  obs field: {obs_field}")

print(f"\n✅ Collected {len(results[model.multires[0]])} batches.")


## 5. Helper: Unnormalisation & Concatenation

Build flat arrays `(N_samples, T, H, W)` with physical units for plotting.

In [ ]:
def unnorm(data: np.ndarray, stats: dict) -> np.ndarray:
    """Apply inverse normalisation (zscore or minmax)."""
    if stats["type"] == "zscore":
        return data * stats["std"] + stats["mean"]
    elif stats["type"] == "minmax":
        return data * (stats["max"] - stats["min"]) + stats["min"]
    return data

def get_stats_for_var(var_suffix: str, res: int) -> dict:
    """Return normalisation stats for a target variable suffix (e.g. 'SIT')."""
    res_key = f"patch_x{res}"
    # Find the source var via var_mapping
    var_mapping_res = model.var_mapping.get(res_key, model.var_mapping) \
                      if isinstance(model.var_mapping, dict) else model.var_mapping
    tgt_var = f"models_{var_suffix}"   # convention used in CROSCIM
    source_var = var_mapping_res.get(tgt_var)  # e.g. 'cristal_SIT'
    if source_var and "_" in source_var:
        group, sv = source_var.split("_", 1)
        return model.norm_stats[group][sv]
    # Fallback to norm_stats_models
    if hasattr(model, "norm_stats_models") and var_suffix in model.norm_stats_models:
        return model.norm_stats_models[var_suffix]
    raise KeyError(f"Cannot find stats for {var_suffix}")


def concat_results(res: int, var_suffix: str, unnormalize: bool = True):
    """
    Concatenate all batch predictions/targets for a given resolution and variable.
    Returns (pred, tgt, obs) each of shape (N*B, T, H, W).
    """
    preds, tgts, obss = [], [], []
    stats = get_stats_for_var(var_suffix, res) if unnormalize else None

    for entry in results[res]:
        p = entry[f"pred_{var_suffix}"].numpy()   # (B, T, H, W)
        t = entry[f"tgt_{var_suffix}"].numpy()
        o = entry.get(f"obs_{var_suffix}", None)
        if o is not None:
            o = o.numpy()

        if unnormalize and stats is not None:
            p = unnorm(p, stats)
            t = unnorm(t, stats)
            if o is not None:
                o = unnorm(o, stats)

        preds.append(p)
        tgts.append(t)
        if o is not None:
            obss.append(o)

    pred = np.concatenate(preds, axis=0)   # (N, T, H, W)
    tgt  = np.concatenate(tgts,  axis=0)
    obs  = np.concatenate(obss,  axis=0) if obss else None
    return pred, tgt, obs

# Quick check
res0 = model.multires[0]
tgt_vars0 = model._get_target_vars_for_resolution(res0)
var0 = tgt_vars0[0].split("_", 1)[1]  # e.g. 'SIT'
p, t, o = concat_results(res0, var0, unnormalize=True)
print(f"Variable: {var0}  |  res=x{res0}")
print(f"  pred shape : {p.shape}")
print(f"  tgt  shape : {t.shape}")
print(f"  obs  shape : {o.shape if o is not None else 'N/A'}")
print(f"  pred range : [{np.nanmin(p):.3f}, {np.nanmax(p):.3f}]")
print(f"  tgt  range : [{np.nanmin(t):.3f}, {np.nanmax(t):.3f}]")
if o is not None:
    nan_pct = 100.0 * np.isnan(o).sum() / o.size
    print(f"  obs  range : [{np.nanmin(o):.3f}, {np.nanmax(o):.3f}]")
    print(f"  obs  NaN % : {nan_pct:.1f}%")


## 6. Spatial Maps — Prediction (`pred_SIT`) | Ground Truth (`models_SIT`) | Observation | Residual

In [ ]:
def plot_spatial(var_suffix: str, res: int,
                 sample_idx: int = 0, time_idx: int = 0,
                 unnormalize: bool = True, cmap: str = "viridis"):
    """Side-by-side: Prediction | Ground Truth | Observation | Residual."""
    pred, tgt, obs = concat_results(res, var_suffix, unnormalize=unnormalize)
    n = pred.shape[0]
    sample_idx = min(sample_idx, n - 1)
    time_idx   = min(time_idx,   pred.shape[1] - 1)

    p   = pred[sample_idx, time_idx].copy()
    t   = tgt [sample_idx, time_idx]

    # Apply GT mask to prediction (wherever GT is NaN, mask prediction too)
    gt_mask = np.isnan(t)
    p[gt_mask] = np.nan

    r   = p - t   # residual

    vmin = np.nanpercentile(t, 2)
    vmax = np.nanpercentile(t, 98)

    ncols = 4 if obs is not None else 3
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5))

    axes[0].imshow(p, vmin=vmin, vmax=vmax, cmap=cmap, origin="lower", aspect="auto", interpolation="nearest")
    axes[0].set_title(f"Prediction (pred_{var_suffix})\n(sample={sample_idx}, t={time_idx})")

    im = axes[1].imshow(t, vmin=vmin, vmax=vmax, cmap=cmap, origin="lower", aspect="auto", interpolation="nearest")
    axes[1].set_title(f"Ground Truth (models_{var_suffix})")
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    rv = np.nanpercentile(np.abs(r), 98)
    rim = axes[2].imshow(r, vmin=-rv, vmax=rv, cmap="RdBu_r", origin="lower", aspect="auto", interpolation="nearest")
    axes[2].set_title("Residual (pred − tgt)")
    plt.colorbar(rim, ax=axes[2], fraction=0.046, pad=0.04)

    if obs is not None:
        o = obs[sample_idx, time_idx]
        axes[3].imshow(o, vmin=vmin, vmax=vmax, cmap=cmap, origin="lower", aspect="auto", interpolation="nearest")
        axes[3].set_title("Observation (input)")

    fig.suptitle(f"{var_suffix}  |  res=x{res}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Show several samples (not just sample_idx=0) — evenly spaced across all
# collected test patches, so we can eyeball variability instead of judging
# the model off a single cherry-picked example.
N_SAMPLES_TO_SHOW = 10

for _res in model.multires:
    for _v in model._get_target_vars_for_resolution(_res):
        _suf = _v.split("_", 1)[1] if "_" in _v else _v
        _pred, _tgt, _obs = concat_results(_res, _suf, unnormalize=True)
        _n = _pred.shape[0]
        _n_show = min(N_SAMPLES_TO_SHOW, _n)
        _sample_idxs = np.linspace(0, _n - 1, _n_show, dtype=int)
        print(f"\n=== {_suf}  x{_res}  —  {_n_show}/{_n} samples (idx={list(_sample_idxs)}) ===")
        for _sidx in _sample_idxs:
            plot_spatial(_suf, _res, sample_idx=int(_sidx), time_idx=0)


## 7. Error Maps & RMSE Spatial Distribution

In [ ]:
def plot_error_maps(var_suffix: str, res: int, unnormalize: bool = True):
    """RMSE map and bias map averaged over all samples and time steps."""
    pred, tgt, _ = concat_results(res, var_suffix, unnormalize=unnormalize)
    residual = pred - tgt                              # (N, T, H, W)
    rmse_map = np.sqrt(np.nanmean(residual ** 2, axis=(0, 1)))   # (H, W)
    bias_map = np.nanmean(residual, axis=(0, 1))                  # (H, W)
    std_map  = np.nanstd(residual,  axis=(0, 1))                  # (H, W)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    im0 = axes[0].imshow(rmse_map, cmap="hot_r", origin="lower", aspect="auto", interpolation="nearest")
    axes[0].set_title("RMSE map")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    rv = np.nanpercentile(np.abs(bias_map), 98)
    im1 = axes[1].imshow(bias_map, vmin=-rv, vmax=rv, cmap="RdBu_r",
                          origin="lower", aspect="auto", interpolation="nearest")
    axes[1].set_title("Bias map (mean residual)")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    im2 = axes[2].imshow(std_map, cmap="viridis", origin="lower", aspect="auto", interpolation="nearest")
    axes[2].set_title("Std of residual")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(f"Error maps — {var_suffix}  |  res=x{res}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"  Global RMSE : {np.nanmean(rmse_map):.4f}")
    print(f"  Global bias : {np.nanmean(bias_map):.4f}")

# All resolutions × variables
for _res in model.multires:
    for _v in model._get_target_vars_for_resolution(_res):
        _suf = _v.split("_", 1)[1] if "_" in _v else _v
        plot_error_maps(_suf, _res)


## 8. Per-Variable Metrics & Time Series

$$\text{RMSE}(t) = \sqrt{\frac{1}{N}\sum_{i}(\hat{x}_i(t) - x_i(t))^2}, \quad \mu = 1 - \frac{\text{RMSE}}{\sigma_{\text{tgt}}}$$

In [ ]:
def compute_metrics(var_suffix: str, res: int, unnormalize: bool = True) -> pd.DataFrame:
    """Return per-timestep RMSE, MAE, and μ-score as a DataFrame."""
    pred, tgt, _ = concat_results(res, var_suffix, unnormalize=unnormalize)
    # pred, tgt: (N, T, H, W)
    T = pred.shape[1]
    records = []
    for t in range(T):
        p_t = pred[:, t].ravel()
        g_t = tgt [:, t].ravel()
        mask = np.isfinite(p_t) & np.isfinite(g_t)
        if mask.sum() == 0:
            records.append({"timestep": t, "RMSE": np.nan, "MAE": np.nan, "mu": np.nan})
            continue
        rmse = np.sqrt(np.mean((p_t[mask] - g_t[mask]) ** 2))
        mae  = np.mean(np.abs(p_t[mask] - g_t[mask]))
        mu   = 1.0 - rmse / (np.std(g_t[mask]) + 1e-8)
        records.append({"timestep": t, "RMSE": rmse, "MAE": mae, "mu": mu})
    return pd.DataFrame(records).set_index("timestep")


def plot_metrics(unnormalize: bool = True):
    """Plot RMSE / MAE / μ per timestep for every variable × resolution."""
    all_vars = set()
    for res in model.multires:
        for v in model._get_target_vars_for_resolution(res):
            all_vars.add((res, v.split("_", 1)[1] if "_" in v else v))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    colors = plt.cm.tab10.colors

    summary_rows = []
    for i, (res, var_suffix) in enumerate(sorted(all_vars)):
        df = compute_metrics(var_suffix, res, unnormalize=unnormalize)
        label = f"{var_suffix} x{res}"
        c = colors[i % len(colors)]
        axes[0].plot(df.index, df["RMSE"], marker="o", color=c, label=label)
        axes[1].plot(df.index, df["MAE"],  marker="o", color=c, label=label)
        axes[2].plot(df.index, df["mu"],   marker="o", color=c, label=label)
        summary_rows.append({
            "var": var_suffix, "res": res,
            "mean_RMSE": df["RMSE"].mean(),
            "mean_MAE":  df["MAE"].mean(),
            "mean_mu":   df["mu"].mean(),
        })

    for ax, title in zip(axes, ["RMSE", "MAE", "μ-score"]):
        ax.set_xlabel("Timestep")
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.suptitle("Per-timestep metrics", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame(summary_rows)
    print("\nSummary table:")
    display(summary.round(4))

plot_metrics(unnormalize=True)


## 9. Diagnostic — Dataloader brut (non-simple) de la xp originale, résolution x50

L'inférence via le dataloader `_simplify` (ci-dessus) est correcte — le souci
vient donc du dataloader **brut** (`BaseDataModuleMultiRes` /
`XrDatasetSupervised`, lecture directe CIMR/CRISTAL avec `domain_limits` /
masque / `reference_source`), utilisé par la xp originale
`xp=CROSCIM/UNet_solvers/base_arctic_croscim_test_sit_UOAI_supervised_forecast`.

On charge cette config séparément, on regarde ce que produit son dataloader de
test en x50 pour `cristal_SIT`, puis on fait passer le **même modèle**
(déjà chargé plus haut) sur ce batch brut pour voir `pred_SIT` en sortie.

In [ ]:
# ── Load the RAW (non-simple) xp config ───────────────────────────────
XP_NAME_RAW = "CROSCIM/UNet_solvers/base_arctic_croscim_test_sit_UOAI_supervised_forecast"

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=CONFIG_DIR, version_base=None):
    cfg_raw = compose(
        config_name="main",
        overrides=[
            f"xp={XP_NAME_RAW}",
            f"domain={DOMAIN}",
        ],
    )

print(f"datamodule._target_ = {cfg_raw.datamodule._target_}")

# ── Instantiate & setup (this opens the raw CIMR/CRISTAL files — can be slow) ──
dm_raw = instantiate(cfg_raw.datamodule)
dm_raw.setup("test")

test_dl_raw = dm_raw.test_dataloader()   # dict: {"patch_x50": DataLoader, "patch_x10": DataLoader}
print("test_dataloader() keys:", list(test_dl_raw.keys()))

dl50_raw = test_dl_raw["patch_x50"]
raw_batch50 = next(iter(dl50_raw))
print("\nRaw x50 batch fields:", raw_batch50._fields)
for field in raw_batch50._fields:
    val = getattr(raw_batch50, field)
    if isinstance(val, torch.Tensor):
        print(f"  {field}: {tuple(val.shape)}")


### 9.1 Ce que produit le dataloader brut — `cristal_SIT` (entrée) et `models_SIT` (cible) en x50

In [ ]:
RAW_SAMPLE_IDX = 0
N_TIME_SHOWN   = 6   # nombre de pas de temps affichés

def plot_raw_dataloader_field(batch, field: str, sample_idx: int = 0, n_time: int = 6, cmap: str = "viridis"):
    """Grille de n_time pas de temps pour un champ NamedTuple du batch, tel que
    produit par le dataloader (valeurs normalisées, pas de unnorm — on regarde
    la donnée brute telle qu'elle arrive dans le modèle)."""
    val = getattr(batch, field)[sample_idx].detach().cpu().numpy()   # (T, H, W)
    n_time = min(n_time, val.shape[0])
    fig, axes = plt.subplots(1, n_time, figsize=(3.2 * n_time, 3.2))
    if n_time == 1:
        axes = [axes]
    vmin = np.nanpercentile(val, 2)
    vmax = np.nanpercentile(val, 98)
    for t in range(n_time):
        im = axes[t].imshow(val[t], vmin=vmin, vmax=vmax, cmap=cmap, origin="lower", aspect="auto", interpolation="nearest")
        axes[t].set_title(f"t={t}", fontsize=9)
        axes[t].axis("off")
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02)
    fig.suptitle(f"Dataloader brut — {field} (sample={sample_idx}, x50, valeurs normalisées)",
                 fontsize=12, fontweight="bold")
    plt.show()
    nan_pct = 100.0 * np.isnan(val).sum() / val.size
    print(f"  {field}: range=[{np.nanmin(val):.3f}, {np.nanmax(val):.3f}]  NaN={nan_pct:.1f}%")

plot_raw_dataloader_field(raw_batch50, "cristal_SIT", sample_idx=RAW_SAMPLE_IDX, n_time=N_TIME_SHOWN)
plot_raw_dataloader_field(raw_batch50, "models_SIT",  sample_idx=RAW_SAMPLE_IDX, n_time=N_TIME_SHOWN)


### 9.2 Inférence du modèle sur ce batch brut — `pred_SIT` vs `models_SIT` vs `cristal_SIT`

In [ ]:
RAW_TIME_IDX = 7   # pas de temps affiché pour la comparaison spatiale

model.eval()
with torch.no_grad():
    b_raw = model.modify_batch(raw_batch50, 50)
    sbatch_raw = model.format_batch_for_solver(b_raw, include_masks=model.include_masks, res=50)
    out_raw = model(batch=sbatch_raw, res=50)
    out_raw_dict = model.split_tensor_to_dict(out_raw, res=50)

stats_SIT = get_stats_for_var("SIT", 50)

pred_raw = unnorm(out_raw_dict["pred_SIT"].cpu().float().numpy(), stats_SIT)     # (B, T, H, W)
tgt_raw  = unnorm(getattr(b_raw, "models_SIT").cpu().float().numpy(), stats_SIT)
obs_raw  = unnorm(getattr(b_raw, "cristal_SIT").cpu().float().numpy(), stats_SIT)

p = pred_raw[RAW_SAMPLE_IDX, RAW_TIME_IDX].copy()
t = tgt_raw [RAW_SAMPLE_IDX, RAW_TIME_IDX]
o = obs_raw [RAW_SAMPLE_IDX, RAW_TIME_IDX]

gt_mask = np.isnan(t)
p[gt_mask] = np.nan
r = p - t

vmin = np.nanpercentile(t, 2)
vmax = np.nanpercentile(t, 98)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
axes[0].imshow(p, vmin=vmin, vmax=vmax, cmap="viridis", origin="lower", aspect="auto", interpolation="nearest")
axes[0].set_title(f"Prediction (pred_SIT)\nxp brute, sample={RAW_SAMPLE_IDX}, t={RAW_TIME_IDX}")

im = axes[1].imshow(t, vmin=vmin, vmax=vmax, cmap="viridis", origin="lower", aspect="auto", interpolation="nearest")
axes[1].set_title("Ground Truth (models_SIT)")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

rv = np.nanpercentile(np.abs(r), 98)
rim = axes[2].imshow(r, vmin=-rv, vmax=rv, cmap="RdBu_r", origin="lower", aspect="auto", interpolation="nearest")
axes[2].set_title("Residual (pred − tgt)")
plt.colorbar(rim, ax=axes[2], fraction=0.046, pad=0.04)

axes[3].imshow(o, vmin=vmin, vmax=vmax, cmap="viridis", origin="lower", aspect="auto", interpolation="nearest")
axes[3].set_title("Observation (cristal_SIT)")

fig.suptitle("SIT | res=x50 | dataloader BRUT (xp originale)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"pred range: [{np.nanmin(p):.3f}, {np.nanmax(p):.3f}]")
print(f"tgt  range: [{np.nanmin(t):.3f}, {np.nanmax(t):.3f}]")
print(f"obs  range: [{np.nanmin(o):.3f}, {np.nanmax(o):.3f}]  (NaN={100*np.isnan(o).sum()/o.size:.1f}%)")
